# Pipeline mimiSBM

Ce notebook a pour étapes : (`import` → `chargement des matrices d'adjacence` → `assemblage du tenseur A` → `estimation mimiSBM`) en utilisant la classe `MimiSBM` fournie.

**Différences importantes avec la version R:**
- La classe Python ne gère que le cas **symétrique** (`SBM_sym`), pas `SBM` dirigé ni `LBM` biparti.
- Elle ne prend **qu'un seul couple (K, Q)** par appel — pas de recherche automatique sur plusieurs valeurs comme dans certaines variantes R.
- Il n'y a pas de paramètre `n_init` intégré : on reproduit ce comportement en relançant `fit()` manuellement plusieurs fois avec des `random_state` différents et en gardant le run au meilleur ELBO (exactement le rôle que jouait `n_init=10` côté R).
- Le critère de sélection de modèle est l'**ELBO** (pas d'ICL calculé nativement ici).

Dépendances nécessaires : `numpy`, `pandas`, `scipy`, `scikit-learn`, `fastkmeanspp`.
```
pip install numpy pandas scipy scikit-learn fastkmeanspp
```


## Imports

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import pickle


# Importation de la classe MimiSBM depuis le fichier mimisbm-main/mimisbm/_model.py
from mimisbm_main.mimisbm._base import MimiSBM

## 1. Chargement des données

In [2]:
data_dir = Path(r"/mnt/c/Users/thoma/OneDrive/Documents/ENSIIE/Master/Stage 3A/Données")
python_adj_dir = data_dir / "adjacency"
plots_dir = Path(r"/mnt/c/Users/thoma/OneDrive/Documents/ENSIIE/Master/Stage 3A/Plots")
plots_dir.mkdir(parents=True, exist_ok=True)


## 2. Chargement des matrices d'adjacence

In [3]:
family_names = ["PPI", "Pfam", "Ks_et_Ka", "TE"]

adj_dict = {}
for fam in family_names:
    path = python_adj_dir / f"adjacency_matrix_{fam}.csv"
    # index_col=0 retire la colonne d'index (equivalent au skip=1 + [, -1] du code R)
    adj = pd.read_csv(path, index_col=0).values

    unique_vals = np.unique(adj)
    print(f"[{fam}] Shape: {adj.shape[0]}x{adj.shape[1]}, "
          f"valeurs uniques: {sorted(unique_vals.tolist())}")

    if not np.all(np.isin(adj, [0, 1])):
        raise ValueError(f"Erreur : {fam} contient des valeurs differentes de 0 et 1")

    adj_dict[fam] = adj.astype(np.int64)


[PPI] Shape: 444x444, valeurs uniques: [0, 1]
[Pfam] Shape: 444x444, valeurs uniques: [0, 1]
[Ks_et_Ka] Shape: 444x444, valeurs uniques: [0, 1]
[TE] Shape: 444x444, valeurs uniques: [0, 1]


## 3. Assemblage et vérifications du tenseur A (N x N x V)

Diagonale mise à 0, symétrisation de sécurité si besoin, 
puis vérifications (carré, dimensions cohérentes, symétrie, diagonale nulle).


In [4]:
N = adj_dict[family_names[0]].shape[0]
V = len(family_names)

assert all(adj_dict[fam].shape[0] == N for fam in family_names)
assert all(adj_dict[fam].shape[1] == N for fam in family_names)

A = np.zeros((N, N, V), dtype=np.int64)
for i, fam in enumerate(family_names):
    adj = adj_dict[fam].copy()
    np.fill_diagonal(adj, 0)  # pas d'auto-boucle
    if not np.array_equal(adj, adj.T):
        adj = np.maximum(adj, adj.T)  # symetrisation de securite
    A[:, :, i] = adj

assert A.shape == (N, N, V)
assert all(np.array_equal(A[:, :, v], A[:, :, v].T) for v in range(V))
assert all(np.all(np.diag(A[:, :, v]) == 0) for v in range(V))

tensor_path = plots_dir / "adjacency_tensor_from_python.npy"
np.save(tensor_path, A)
print(f"Tenseur A sauvegarde : {tensor_path}")


Tenseur A sauvegarde : /mnt/c/Users/thoma/OneDrive/Documents/ENSIIE/Master/Stage 3A/Plots/adjacency_tensor_from_python.npy


## 4. Estimation mimiSBM (multi-run, équivalent de `n_init = 10` en R)

La classe Python n'a pas de paramètre `n_init` intégré (contrairement à la fonction
R `mimiSBM()`). On reproduit donc plusieurs initialisations aléatoires indépendantes, on garde le run qui atteint le meilleur ELBO.

In [5]:
K_set = 4  # nombre de clusters d'individus a tester
Q_set = 1  # nombre de composantes de vues a tester
N_INIT = 10

best_model = None
best_elbo = -np.inf

for run in range(1, N_INIT + 1):
    model = MimiSBM(
        n_clusters=K_set,
        n_components=Q_set,
        clusters_prior="jeffreys",
        components_prior="jeffreys",
        adjacency_prior="jeffreys",
        max_iter=100,
        tol=1e-4,
        random_state=run,  # graine differente a chaque run, comme les 10 runs R
    )
    model.fit(A)
    print(f"------------ Run {run} ------------")
    print(f"Evidence Lower Bound: {model.elbo_:.2f}  (converged={model.converged_})")

    if model.elbo_ > best_elbo:
        best_elbo = model.elbo_
        best_model = model

print()
print(f"Meilleur run retenu : ELBO = {best_elbo:.2f}")


------------ Run 1 ------------
Evidence Lower Bound: -250163.85  (converged=True)
------------ Run 2 ------------
Evidence Lower Bound: -248987.48  (converged=True)
------------ Run 3 ------------
Evidence Lower Bound: -246076.02  (converged=True)
------------ Run 4 ------------
Evidence Lower Bound: -250070.67  (converged=True)
------------ Run 5 ------------
Evidence Lower Bound: -250327.18  (converged=True)
------------ Run 6 ------------
Evidence Lower Bound: -250032.68  (converged=True)
------------ Run 7 ------------
Evidence Lower Bound: -246076.02  (converged=True)
------------ Run 8 ------------
Evidence Lower Bound: -249855.36  (converged=True)
------------ Run 9 ------------
Evidence Lower Bound: -248485.64  (converged=True)
------------ Run 10 ------------
Evidence Lower Bound: -247280.87  (converged=True)

Meilleur run retenu : ELBO = -246076.02


## 5. Résultats

Équivalent du bloc R final : extraction des clusters d'individus, des composantes de vues, tailles de clusters.

In [6]:
node_clusters, view_components = best_model.predict()

print("Tailles des clusters d'individus (K):")
print(pd.Series(node_clusters).value_counts().sort_index())

print()
print("Composante de vue retenue par famille de descripteurs:")
print(pd.DataFrame({"vue": family_names, "composante": view_components[:len(family_names)]}))


Tailles des clusters d'individus (K):
0    107
1     89
2    101
3    147
Name: count, dtype: int64

Composante de vue retenue par famille de descripteurs:
        vue  composante
0       PPI           0
1      Pfam           0
2  Ks_et_Ka           0
3        TE           0


In [7]:
model_path = plots_dir / "mimiSBM_model_from_python.pkl"
with open(model_path, "wb") as f:
    pickle.dump(best_model, f)

print(f"Modele mimiSBM sauvegarde : {model_path}")


Modele mimiSBM sauvegarde : /mnt/c/Users/thoma/OneDrive/Documents/ENSIIE/Master/Stage 3A/Plots/mimiSBM_model_from_python.pkl


In [8]:
# boucle sur K et sur Q pour trouver le meilleur couple (K, Q) selon l'ELBO
best_model = None
best_elbo = -np.inf
for K in range(2, 6):  # K de 2 a 5
    for Q in range(2, 5):  # Q de 2 a 4
        for run in range(1, N_INIT + 1):
            model = MimiSBM(
                n_clusters=K,
                n_components=Q,
                clusters_prior="jeffreys",
                components_prior="jeffreys",
                adjacency_prior="jeffreys",
                max_iter=100,
                tol=1e-4,
                random_state=run,
            )
            model.fit(A)

            if model.elbo_ > best_elbo:
                best_elbo = model.elbo_
                best_model = model

        print(f"Meilleur run retenu - K={K}, Q={Q} : ELBO = {best_elbo:.2f}")
        node_clusters, view_components = best_model.predict()

        print(f"Tailles des clusters d'individus (K = {K}):")
        print(pd.Series(node_clusters).value_counts().sort_index())

        print("Composante de vue retenue par famille de descripteurs:")
        print(pd.DataFrame({"vue": family_names, "composante": view_components[:len(family_names)]}))

        # printer responsibilities du model pour chaque composante de vue
        print("Responsibilities du model pour chaque composante de vue:")
        print(best_model.component_responsibilities_)

        # print uniq de composantes de vue retenues
        print("Composantes de vue retenues (uniques):")
        print(np.unique(view_components[:len(family_names)]))

        print("Clusters uniques d'individus (K):")
        print(np.unique(node_clusters))

Meilleur run retenu - K=2, Q=2 : ELBO = -203839.39
Tailles des clusters d'individus (K = 2):
0    147
1    297
Name: count, dtype: int64
Composante de vue retenue par famille de descripteurs:
        vue  composante
0       PPI           1
1      Pfam           1
2  Ks_et_Ka           0
3        TE           1
Responsibilities du model pour chaque composante de vue:
[[0. 1.]
 [0. 1.]
 [1. 0.]
 [0. 1.]]
Composantes de vue retenues (uniques):
[0 1]
Clusters uniques d'individus (K):
[0 1]
Meilleur run retenu - K=2, Q=3 : ELBO = -201949.38
Tailles des clusters d'individus (K = 2):
0    147
1    297
Name: count, dtype: int64
Composante de vue retenue par famille de descripteurs:
        vue  composante
0       PPI           1
1      Pfam           0
2  Ks_et_Ka           2
3        TE           0
Responsibilities du model pour chaque composante de vue:
[[0. 1. 0.]
 [1. 0. 0.]
 [0. 0. 1.]
 [1. 0. 0.]]
Composantes de vue retenues (uniques):
[0 1 2]
Clusters uniques d'individus (K):
[0 1]
Meil

## 6. Points d'attention (Python vs R)

- **Symétrie forcée** : `_validate_A` force `A |= A.swapaxes(0, 1)` et caste en booléen —
  toute pondération/poids continu serait perdu. Si tes matrices Python sont déjà 0/1
  symétriques (ce qui est le cas ici), ce n'est pas un problème.
- **Pas de recherche automatique de K/Q** : contrairement à la question du collapse
  observé côté R (K=4,Q=2 demandés → K=2,Q=1 effectifs), la version Python ne renvoie
  jamais moins de clusters que demandé — les `cluster_responsibilities_`/
  `component_responsibilities_` restent de taille (N, K) et (V, Q) fixes. Il est donc
  utile de comparer les tailles de clusters (`value_counts` ci-dessus) : un cluster
  quasi vide est le signe Python du même phénomène de collapse observé en R.
- **`n_init` manuel** : si tu veux paralléliser les runs (comme `nbCores` côté R), tu
  peux remplacer la boucle `for run in range(...)` par `joblib.Parallel`.
